# Online Fraud Detection - End-to-End Experiment
This notebook implements the complete machine learning pipeline directly, including data loading, feature engineering, preprocessing, model training with hyperparameter tuning, and evaluation.

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

## 2. Data Ingestion
Loading the dataset. For demonstration, if the dataset doesn't exist, we'll create a dummy dataset with the required schema.

In [ ]:
df=pd.read_csv()

## 3. Feature Engineering
Creating derived features that capture the discrepancies in balances, which are strong indicators of fraud.

In [ ]:
def create_derived_features(dataframe):
    df_copy = dataframe.copy()
    # Calculate balance errors
    df_copy['errorBalanceOrg'] = df_copy['newbalanceOrig'] + df_copy['amount'] - df_copy['oldbalanceOrg']
    df_copy['errorBalanceDest'] = df_copy['oldbalanceDest'] + df_copy['amount'] - df_copy['newbalanceDest']
    return df_copy

df_engineered = create_derived_features(df)
print("Engineered Features Added:")
display(df_engineered[['errorBalanceOrg', 'errorBalanceDest']].head())

## 4. Train-Test Split

In [ ]:
target_column = 'isFraud'
X = df_engineered.drop(columns=[target_column])
y = df_engineered[target_column]

# Check if stratify is possible (needs at least 2 samples per class in dummy data)
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
except ValueError:
    print("Stratified split failed (likely due to too few samples). Using standard split.")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## 5. Data Transformation (Preprocessing)
Building a ColumnTransformer to handle numerical scaling and categorical encoding.

In [ ]:
numerical_columns = X_train.select_dtypes(exclude="object").columns.tolist()
categorical_columns = X_train.select_dtypes(include="object").columns.tolist()

num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ("scaler", StandardScaler(with_mean=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, numerical_columns),
    ("cat", cat_pipeline, categorical_columns)
])

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"Transformed X_train shape: {X_train_transformed.shape}")

## 6. Model Training & Hyperparameter Tuning
We evaluate multiple models using GridSearchCV to find the best estimator based on ROC-AUC.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(),
    "RandomForestClassifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

# Parameters for GridSearch
params = {
    "LogisticRegression": {
        "C": [0.01, 0.1, 1],
        "solver": ['liblinear']
    },
    "RandomForestClassifier": {
        "n_estimators": [50, 100],
        "max_depth": [5, 10]
    },
    "XGBClassifier": {
        "n_estimators": [50, 100],
        "learning_rate": [0.01, 0.1],
        "max_depth": [3, 5]
    }
}

best_models = {}
model_scores = {}

# Since dummy data might be too small for CV=3, handle dynamically
cv_folds = min(3, len(y_train) // 2) 
cv_folds = max(2, cv_folds) # ensure at least 2 folds

for model_name, model in models.items():
    print(f"Training {model_name}...")
    gs = GridSearchCV(model, params[model_name], cv=cv_folds, scoring='roc_auc', n_jobs=-1)
    
    try:
        gs.fit(X_train_transformed, y_train)
        # Retrain on full training data with best params
        best_model = model.set_params(**gs.best_params_)
        best_model.fit(X_train_transformed, y_train)
        
        best_models[model_name] = best_model
        
        # Predict and score on test set
        y_test_pred = best_model.predict_proba(X_test_transformed)[:, 1] if hasattr(best_model, "predict_proba") else best_model.predict(X_test_transformed)
        
        try:
            score = roc_auc_score(y_test, y_test_pred)
        except ValueError:
            # Fallback if only one class is present in y_test
            score = 0.5 
            
        model_scores[model_name] = score
        print(f"  Best Params: {gs.best_params_}")
        print(f"  Test ROC-AUC: {score:.4f}\n")
    except Exception as e:
        print(f"  Error training {model_name}: {e}\n")

if model_scores:
    best_model_name = max(model_scores, key=model_scores.get)
    final_best_model = best_models[best_model_name]
    print(f"==== Best Model: {best_model_name} with ROC-AUC {model_scores[best_model_name]:.4f} ====")
else:
    print("No models were successfully trained.")

## 7. Model Evaluation
Evaluating the best model in depth using Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
if model_scores:
    y_pred = final_best_model.predict(X_test_transformed)
    y_pred_proba = final_best_model.predict_proba(X_test_transformed)[:, 1] if hasattr(final_best_model, "predict_proba") else y_pred

    try:
        roc_auc = roc_auc_score(y_test, y_pred_proba)
    except ValueError:
        roc_auc = 0.5
        
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    metrics = {
        "ROC-AUC": roc_auc,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1
    }

    print("Final Model Metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")